# 02 · ndarray 핵심 & NumPy → CuPy 포팅

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

이 노트북은 코스의 **레퍼런스 노트북**입니다. NumPy `ndarray`의 구조를 이해하고,
뷰/복사·브로드캐스팅·축 연산을 익힌 뒤, **`import numpy as np` → `import cupy as cp`** 만으로
GPU에서 동작시키는 "드롭인" 포팅을 실습합니다.

## 학습 목표
- `ndarray`의 4요소(**data·dtype·shape·strides**)와 뷰 vs 복사를 설명한다.
- 축(axis) 기반 집계와 **브로드캐스팅**(stretch 규칙)을 능숙하게 쓴다.
- NumPy 코드를 CuPy로 포팅하고, **장치 비종속(agnostic) 코드**를 작성한다.
- **암묵적 전송**과 디바이스 관리의 주의점을 안다.

## 목차
1. [NumPy → CuPy: 드롭인 대체](#1)
2. [ndarray의 구조 (anatomy)](#2)
3. [뷰(View) vs 복사(Copy)](#3)
4. [축(axis)과 집계](#4)
5. [브로드캐스팅: "stretch" 규칙](#5)
6. [장치 비종속 코드 (NumPy dispatch)](#6)
7. [암묵적 전송 주의](#7)
8. [디바이스 관리](#8)
9. [연습문제](#9)
10. [체크포인트](#10)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
import cupyx as cpx
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. NumPy → CuPy: 드롭인 대체

CuPy는 **NumPy API를 GPU에서** 구현합니다(백엔드는 CUDA C++). NumPy를 알면 CuPy를 이미 아는 셈입니다.
많은 경우 **import 한 줄**만 바꿔도 GPU에서 돌아갑니다.

<img src="images/figures/new_cupy_speedup.png" width="560">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
# 같은 (50,500,500) 배열 생성을 CPU vs GPU로 비교 (~95MB)
shape = (50, 500, 500)
compare('ones'+str(shape), lambda: np.ones(shape), lambda: cp.ones(shape), n_repeat=10)

# 타입과 위치 확인
g = cp.ones(shape)
print('type  :', type(g))
print('device:', g.device)

<a id="2"></a>
## 2. ndarray의 구조 (anatomy)

📖 [`cupy.ndarray` 레퍼런스](https://docs.cupy.dev/en/stable/reference/ndarray.html) — CuPy 공식 overview의 **N차원 배열** 구성요소.
지원 dtype: `bool_`, 정수(`int8~64`, `uint8~64`), 실수(`float16/32/64`), 복소수(`complex64/128`). 인덱싱·고급 인덱싱·브로드캐스팅 의미는 `numpy.ndarray`와 동일합니다.

`ndarray`는 파이썬 리스트와 달리 **연속된 고정 크기 메모리 블록**입니다. 네 가지 핵심 속성이 효율의 비결입니다.
- **data**: 원소들이 저장된 메모리 블록에 대한 포인터
- **dtype**: 모든 원소의 균일한 자료형(예: `float32`)
- **shape**: 각 차원의 크기 튜플(예: `(4, 3)`)
- **strides**: 다음 원소로 가기 위한 **바이트 수**(뷰/슬라이싱의 핵심)

<img src="images/figures/new_ndarray_anatomy.png" width="620">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
# 큰 배열로 '밀집 메모리'를 체감 (GPU에 생성)
N = 50_000_000
arr = cp.arange(1, N + 1, dtype=cp.float32)   # 1..N
print('dtype :', arr.dtype)
print('ndim  :', arr.ndim)
print('size  :', f'{arr.size:,}')
print('shape :', arr.shape)
print('nbytes:', f'{arr.nbytes/2**30:.3f} GB')

# reshape는 메타데이터(shape/strides)만 바꾸는 '뷰' (데이터 복사 X)
M = arr.reshape(-1, 5)
print('reshape shape :', M.shape)
print('reshape strides:', M.strides, '(바이트 단위)')

**논리(2D 인덱스) ↔ 물리(1차원 저장) 매핑**: 다차원 인덱스는 결국 1차원 메모리로 펼쳐 저장되며, 그 **순서를 결정하는 것이 strides**입니다.
NumPy/CuPy 기본은 **행 우선(C order)** 입니다. (아래 그림은 매핑 개념을 보여주는 *열 우선* 예시)

<img src="images/figures/new_logical_vs_storage.png" width="660">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

<a id="3"></a>
## 3. 뷰(View) vs 복사(Copy)

슬라이싱·`reshape`·전치(`.T`)는 보통 **뷰**를 반환합니다 — 메타데이터만 바뀌고 물리 메모리는 공유하므로 거의 즉시.
반면 `A + B`, `A * 5` 같은 연산은 **새 배열(복사)** 를 할당합니다. 순차 연산이 많으면 복사가 성능 함정이 됩니다.
아래 슬라이싱 도해에서 노란 테두리가 "원본의 뷰"입니다.

<img src="images/figures/new_views_slicing.png" width="680">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
a = cp.arange(12, dtype=cp.float32).reshape(3, 4)
v = a[:, :2]            # 슬라이싱 -> 뷰
c = a[:, :2].copy()     # 명시적 복사
r = a.reshape(2, 6)     # reshape -> 뷰
print('슬라이싱 뷰? (메모리 공유):', v.data.ptr == a.data.ptr)
print('복사본?    (메모리 분리):', c.data.ptr == a.data.ptr)
print('reshape 뷰?            :', r.data.ptr == a.data.ptr)

**순차 연산의 복사 함정**: 아래 `seq`의 각 줄(`x*5`, `x*x`, `x+x`)은 매번 새 배열을 만듭니다.
그래도 GPU에 머무는 한 host 전송은 없고, CuPy는 필요할 때만 GPU→CPU로 가져옵니다.

In [ ]:
# GPU에서 복사를 일으키는 코드. 매번 새로운 배열을 만들어서 GPU 메모리를 낭비함.
def seq(x):
    x = x * 5
    x = x * x
    x = x + x
    return x

x_gpu = cp.ones((50, 500, 500), dtype=cp.float32)
print_bench(bench(lambda: seq(x_gpu), n_repeat=10, name='sequential(copies)'))

In [ ]:
# 복사 함정을 해결한 올바른 코드
def seq_efficient(x) :
    x *= 5   # 새로운 배열을 만들지 않고, 원래 x가 있던 자리에 5를 곱해버림!
    x *= x   # 원래 자리에 그대로 제곱을 해버림!
    x += x   # 원래 자리에 그대로 더해버림!
    return x

x_gpu = cp.ones((50, 500, 500), dtype=cp.float32)
print_bench(bench(lambda: seq(x_gpu), n_repeat=10, name='sequential(copies)'))
print_bench(bench(lambda: seq_efficient(x_gpu), n_repeat=10, name='sequential(seq_efficient)'))

<a id="4"></a>
## 4. 축(axis)과 집계

`sum/mean/max` 등 집계는 **접을(reduce) 축**을 지정합니다.
- `axis=0`: 첫 번째 차원(행)을 접음 → **열별** 결과
- `axis=1`: 두 번째 차원(열)을 접음 → **행별** 결과
- `keepdims=True`: 차원 수를 보존(브로드캐스팅에 유용)

<img src="images/figures/new_array_functions_axis.png" width="640">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
M = cp.arange(1, 13, dtype=cp.float32).reshape(3, 4)
print('M =\n', cp.asnumpy(M))
print('axis=0 (열별 합):', cp.asnumpy(M.sum(axis=0)))
print('axis=1 (행별 합):', cp.asnumpy(M.sum(axis=1)))
print('keepdims shape :', M.sum(axis=1, keepdims=True).shape)

<a id="5"></a>
## 5. 브로드캐스팅: "stretch" 규칙

서로 다른 모양의 배열 간 연산 시, 작은 쪽을 "늘려(stretch)" 맞춥니다. **호환 규칙**: 두 차원이
(1) 같거나 (2) 한쪽이 1이면 호환. 1인 차원은 **메모리 복사 없이** 논리적으로 확장됩니다.

<img src="images/figures/new_broadcasting_ok.png" width="520">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

<img src="images/figures/new_broadcasting_err.png" width="520">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
# 행별 정규화: (4,5)를 행 합 (4,1)로 나누면 열 방향으로 stretch
M = cp.random.random((4, 5), dtype=cp.float32)
row_sums = M.sum(axis=1, keepdims=True)     # (4,1)
Mn = M / row_sums                            # 브로드캐스팅
print('정규화 후 각 행 합:', cp.asnumpy(Mn.sum(axis=1)))   # ~1.0

# 호환되지 않는 경우
try:
    _ = cp.ones((3, 2)) + cp.arange(3)
except ValueError as e:
    print('ValueError:', e)

<a id="6"></a>
## 6. 장치 비종속 코드 (NumPy dispatch)

`cp.get_array_module(x)` 는 입력이 NumPy면 `numpy`, CuPy면 `cupy` 모듈을 돌려줍니다.
이를 쓰면 **CPU/GPU 양쪽에서 동작하는 함수** 하나를 작성할 수 있습니다.
또한 많은 `np.*` 함수는 `__array_function__`(NEP 18) 덕분에 **CuPy 배열을 넣으면 자동으로 GPU에서 실행**됩니다.

### NumPy ↔ CuPy 자동 디스패치 흐름 (`__array_function__` protocol)

1. **사용자 코드**
   `np.linalg.svd(x_gpu)` 호출

2. **NumPy 내부**
   입력이 CuPy 배열임을 감지(`__array_function__` protocol) → 자체 연산 포기. Cupy 구현으로 위임(dispatch)

3. **CuPy 내부**
   "바톤 터치!" → `cp.linalg.svd(x_gpu)` 호출로 위임

4. **GPU 하드웨어**
   CuPy가 준비한 CUDA 커널(cuSOLVER 라이브러리) 실행

> 핵심: NumPy 함수를 그대로 호출해도, 입력이 CuPy 배열이면 NumPy가 알아서 CuPy 구현으로 위임(dispatch)합니다.

In [ ]:
def standardize(x):
    # 1. 입력 x가 NumPy 배열이면 np를, CuPy 배열이면 cp를 반환합니다.
    xp = cp.get_array_module(x)     
    mean = xp.mean(x)
    std = xp.std(x)    
    return (x - mean) / (std + 1e-8)

a_np = np.random.randn(100_000).astype(np.float32)
print('NumPy 입력 -> 결과 type:', type(standardize(a_np)))
print('CuPy  입력 -> 결과 type:', type(standardize(cp.asarray(a_np))))

# np.* 함수가 CuPy 배열로 디스패치되어 GPU에서 실행되는 예 (SVD, O(N^3))
x_gpu = cp.random.random((1500, 600), dtype=cp.float32)
print_bench(bench(lambda: np.linalg.svd(x_gpu, compute_uv=False), n_repeat=3, name='np.linalg.svd→GPU'))

<a id="7"></a>
## 7. 암묵적 전송 주의

숨은 성능 함정은 **CPU↔GPU 암묵적 전송**입니다. CuPy는 일부를 막아줍니다:
`np.asarray(gpu_array)`처럼 `__array__`로 host 변환을 시도하면 **조용히 복사하지 않고 `TypeError`** 를 냅니다.
하지만 **출력(print), 스칼라 변환(`float`, `.item()`)** 등은 암묵적으로 GPU→CPU 전송을 일으킵니다.

In [ ]:
g = cp.arange(5, dtype=cp.float32)
try:
    np.asarray(g)            # 암묵적 host 복사 시도 -> 차단
except TypeError as e:
    print('np.asarray(gpu) 차단:', str(e)[:90])

print('명시적 전송 asnumpy :', cp.asnumpy(g))   # 의도적 전송 (OK)
print('스칼라 변환 float()  :', float(g.sum()))   # 암묵적 전송 발생

<a id="8"></a>
## 8. 디바이스 관리

GPU가 여러 개면 `with cp.cuda.Device(i):` 로 특정 장치에 배열을 만들 수 있습니다.
CuPy 연산은 보통 입력들이 **같은 장치**에 있어야 합니다.

In [ ]:
print('현재 device id:', cp.cuda.Device().id)
with cp.cuda.Device(0):
    x = cp.random.random((1000, 1000), dtype=cp.float32)
print('x.device:', x.device)

<a id="9"></a>
## 9. 연습문제 — 장치 비종속 변환 포팅

입력 배열 `x`에 대해 다음을 수행하는 **장치 비종속** 함수 `transform(x)` 를 완성하세요.
1) z-score 표준화 `z = (x - mean) / (std + 1e-8)`
2) `z`를 `(-1, 5)`로 reshape
3) 각 행을 그 행의 **절댓값 최댓값**으로 나눠 반환

조건: `cp.get_array_module`을 써서 NumPy/CuPy 모두에서 동작하게 하고, 같은 입력에 대해 결과가 일치함을 확인하세요.

In [ ]:
def transform(x):
    # TODO: xp = cp.get_array_module(x) 를 사용해 위 1)~3)을 구현
    raise NotImplementedError

# 검증 (구현 후 주석 해제): 같은 host 입력 -> CPU/GPU 결과 일치
x_np = np.random.randn(2_000_000).astype(np.float32)
# ref = transform(x_np)
# out = cp.asnumpy(transform(cp.asarray(x_np)))
# np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('정확성 OK')
# compare('transform', lambda: transform(x_np), lambda: transform(cp.asarray(x_np)), n_repeat=5)

<details>
<summary>💡 해답 보기</summary>

```python
def transform(x):
    xp = cp.get_array_module(x)
    z = (x - xp.mean()) / (xp.std() + 1e-8)
    M = z.reshape(-1, 5)
    return M / xp.abs(M).max(axis=1, keepdims=True)

x_np = np.random.randn(2_000_000).astype(np.float32)
ref = transform(x_np)
out = cp.asnumpy(transform(cp.asarray(x_np)))
np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4)
print('정확성 OK')
compare('transform', lambda: transform(x_np),
        lambda: transform(cp.asarray(x_np)), n_repeat=5)
```

포인트: `xp = cp.get_array_module(x)` 하나로 NumPy/CuPy 양쪽을 지원합니다.
같은 host 입력을 넣으면 결과가 일치하지만, 각자 난수를 생성하면 RNG가 달라 일치하지 않습니다(단원 1 참고).
</details>

## 🧪 추가 연습

**연습 A — 열별 Min-Max 정규화** (브로드캐스팅): 각 열을 `[0,1]`로 정규화하는 장치 비종속 함수를 완성하세요.

In [ ]:
def col_minmax(x):
    # TODO: 열별 min/max로 (x-min)/(max-min) 정규화 (cp.get_array_module 사용)
    raise NotImplementedError

X_np = np.random.randn(1000, 8).astype(np.float32)
# ref = col_minmax(X_np); out = cp.asnumpy(col_minmax(cp.asarray(X_np)))
# np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('OK')

<details><summary>💡 해답 보기</summary>

```python
def col_minmax(x):
    xp = cp.get_array_module(x)
    mn = xp.min(axis=0, keepdims=True)
    mx = xp.max(axis=0, keepdims=True)
    return (x - mn) / (mx - mn + 1e-8)

X_np = np.random.randn(1000, 8).astype(np.float32)
ref = col_minmax(X_np)
out = cp.asnumpy(col_minmax(cp.asarray(X_np)))
np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('OK')
```
</details>

**연습 B — 뷰로 원본 수정**: 슬라이싱 뷰에 대입해 **원본이 바뀌는지** 확인하고, `.copy()`와 대비하세요.
- 슬라이싱 (Slicing): 덩어리에서 일부를 잘라내는 행위. 파이썬에서 a[:2, :2]라고 쓰는 것이 바로 슬라이싱임.
- 뷰 (View): 새로 만든 데이터가 아니라, 원본을 바라보는 주소표 (뷰). 슬라이싱을 했을 때 컴퓨터는 데이터를 새로 복사하지 않고, "원본의 어느 주소부터 어느 주소까지 바라보면 된다"라는 메모리 주소표(View)만 새로 생성함.


In [ ]:
def zero_block(a):
    # TODO: a의 좌상단 2x2를 '뷰'로 0으로 만들어 원본도 바뀌게 하세요 (복사 금지)
    raise NotImplementedError

A = cp.arange(16, dtype=cp.float32).reshape(4, 4)
# print(zero_block(A))

<details><summary>💡 해답 보기</summary>

```python
def zero_block(a):
    v = a[:2, :2]     # 슬라이싱 -> 뷰 (메모리 공유)
    v[:] = 0          # 뷰에 대입하면 원본도 변경
    return a

A = cp.arange(16, dtype=cp.float32).reshape(4, 4)
print(zero_block(A))   # 좌상단 2x2가 0
# 대조: B[:2,:2].copy() 에 대입하면 원본은 그대로
```
</details>

<a id="10"></a>
## 10. 체크포인트

- [ ] ndarray의 data·dtype·shape·strides를 설명할 수 있다
- [ ] 슬라이싱/`reshape`이 뷰인지 복사인지 `data.ptr`로 확인했다
- [ ] `axis`와 `keepdims`로 원하는 집계를 만들 수 있다
- [ ] 브로드캐스팅 호환 규칙(같거나 1)을 적용해 행별 정규화를 했다
- [ ] `cp.get_array_module`로 장치 비종속 함수를 작성했다
- [ ] `np.asarray(gpu)`가 차단되는 이유와 암묵적 전송을 안다

다음: **`03_numpy_routines`** — NumPy 루틴(`cupy.*` 모듈 함수·`linalg`·`fft`·`random`)을 예제로 다룹니다.